# Simple Table Extraction — Eval

Reads `table_catalog.csv`, filters for `agent=general_table` or `agent=simple_table`, runs extraction on each entry with complete ground truth, and prints aggregate accuracy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from shared.client import client, DEFAULT_MODEL
from shared.pdf import render_pages
from shared.eval import compare_tables, print_accuracy_summary
from agents.general_table.extract import extract_tables_from_page, stitch_page_results

In [ ]:
MODEL = DEFAULT_MODEL

catalog = pd.read_csv("../table_catalog.csv")
table_entries = catalog[catalog["agent"].isin(["general_table", "simple_table"])].copy()
# Only eval entries with status=complete (ground truth available)
eval_entries = table_entries[table_entries["status"] == "complete"]
print(f"Table entries to evaluate: {len(eval_entries)}")
eval_entries

In [ ]:
results = []

for _, row in eval_entries.iterrows():
    paper_id = row["paper_id"]
    page = int(row["page"])
    pdf_path = os.path.join("..", "papers", paper_id, "paper.pdf")
    gt_path = os.path.join("..", row["ground_truth_path"])

    print(f"\n--- {paper_id} p{page} ({row['description']}) ---")

    # Render and extract
    images = render_pages(pdf_path, [page])
    page_result = extract_tables_from_page(client, MODEL, images[page])
    page_results = {page: page_result}
    dfs = stitch_page_results(page_results)

    if not dfs:
        print("  No tables extracted!")
        continue

    # Compare first extracted table to ground truth
    df_extracted = dfs[0]
    df_truth = pd.read_csv(gt_path)
    result = compare_tables(df_truth, df_extracted)
    result["paper_id"] = paper_id
    result["page"] = page
    results.append(result)
    print_accuracy_summary(result)

In [ ]:
# Aggregate accuracy
if results:
    total_cells = sum(r["total_cells"] for r in results)
    correct_cells = sum(r["correct_cells"] for r in results)
    print(f"\n{'=' * 60}")
    print(f"AGGREGATE: {correct_cells}/{total_cells} = {correct_cells/total_cells:.1%}")
    print(f"{'=' * 60}")
else:
    print("No entries evaluated.")